# 🧠 Alzheimer's MRI Detection — ResNet18 + BiGRU + Attention (v3)

**Fixes from v2:** Removed class weights & label smoothing (caused catastrophic class collapse). Uses dual-head fusion (CNN + GRU) and clean training.

In [ ]:
# Cell 1: Setup
!pip install -q datasets scikit-learn pillow
import torch, numpy as np
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Model — ResNet18 + BiGRU + Attention (dual-head fusion)
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

CLASS_NAMES = ['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented']

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim//2), nn.Tanh(), nn.Linear(dim//2, 1))
    def forward(self, x):
        return (F.softmax(self.attn(x), dim=1) * x).sum(dim=1)

class AlzheimerHybridModel(nn.Module):
    def __init__(self, num_classes=4, gru_hidden=256, dropout=0.3, pretrained=True):
        super().__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                self.conv1.weight.copy_(resnet.conv1.weight.mean(dim=1, keepdim=True))
        self.bn1 = resnet.bn1; self.relu = resnet.relu; self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1; self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3; self.layer4 = resnet.layer4
        # Head A: CNN global avg pool
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.head_a = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, 256), nn.ReLU(True),
                                     nn.Dropout(dropout*0.5), nn.Linear(256, num_classes))
        # Head B: GRU + Attention
        self.gru = nn.GRU(512, gru_hidden, 1, batch_first=True, bidirectional=True)
        self.attention = AttentionPooling(gru_hidden*2)
        self.head_b = nn.Sequential(nn.LayerNorm(gru_hidden*2), nn.Dropout(dropout),
                                     nn.Linear(gru_hidden*2, 256), nn.ReLU(True),
                                     nn.Dropout(dropout*0.5), nn.Linear(256, num_classes))
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        # Head A
        p = self.gap(x).view(x.size(0), -1)
        la = self.head_a(p)
        # Head B
        B,C,H,W = x.shape
        tokens = x.view(B,C,H*W).permute(0,2,1).contiguous()
        gru_out, _ = self.gru(tokens)
        lb = self.head_b(self.attention(gru_out))
        return 0.6 * la + 0.4 * lb
    def get_last_conv_layer(self):
        return self.layer4[-1].conv2

m = AlzheimerHybridModel(pretrained=True)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
print(f'Test: {m(torch.randn(2,1,128,128)).shape}')
print('✅ Model OK')

In [ ]:
# Cell 3: Load Dataset
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from collections import Counter

print('📥 Loading dataset...')
train_ds = load_dataset('Falah/Alzheimer_MRI', split='train')
test_ds = load_dataset('Falah/Alzheimer_MRI', split='test')
print(f'Train: {len(train_ds)}, Test: {len(test_ds)}')

# Simple, effective augmentation (not too aggressive)
train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomResizedCrop(128, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

class MRIDataset(Dataset):
    def __init__(self, hf_ds, tfm):
        self.ds = hf_ds; self.tfm = tfm
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        ex = self.ds[idx]; img = ex['image']
        if not isinstance(img, Image.Image):
            arr = np.squeeze(np.array(img))
            if arr.dtype != np.uint8: arr = (255*(arr-arr.min())/(arr.max()-arr.min()+1e-8)).astype(np.uint8)
            img = Image.fromarray(arr, mode='L')
        elif img.mode != 'L': img = img.convert('L')
        return self.tfm(img), int(ex['label'])

train_loader = DataLoader(MRIDataset(train_ds, train_transform), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(MRIDataset(test_ds, val_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

counts = Counter(int(x) for x in train_ds['label'])
print(f'Distribution: {dict(sorted(counts.items()))}')
print('✅ Data ready')

In [ ]:
# Cell 4: Training Setup
import time, os
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AlzheimerHybridModel(num_classes=4, pretrained=True).to(DEVICE)

# Plain CrossEntropyLoss — NO class weights, NO label smoothing
criterion = nn.CrossEntropyLoss()

# Single optimizer, simple setup
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
NUM_EPOCHS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler() if DEVICE.type == 'cuda' else None

print(f'Device: {DEVICE}')
print(f'Loss: CrossEntropyLoss (no class weights)')
print('✅ Setup complete')

In [ ]:
# Cell 5: Train!
os.makedirs('checkpoints', exist_ok=True)
best_acc = 0.0; patience = 6; no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train(); rloss = 0; correct = 0; tot = 0; t0 = time.time()
    for imgs, labs in train_loader:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast(device_type='cuda'):
                out = model(imgs); loss = criterion(out, labs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(imgs); loss = criterion(out, labs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        rloss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labs).sum().item()
        tot += labs.size(0)
    scheduler.step()

    # Validate
    model.eval(); vc = 0; vt = 0; vl = 0
    with torch.no_grad():
        for imgs, labs in test_loader:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            out = model(imgs); loss = criterion(out, labs)
            vl += loss.item() * imgs.size(0)
            vc += (out.argmax(1) == labs).sum().item()
            vt += labs.size(0)
    vacc = vc / vt; mark = ''
    if vacc > best_acc:
        best_acc = vacc; no_improve = 0; mark = ' ⭐'
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': vacc, 'num_classes': 4, 'class_names': CLASS_NAMES},
                   'checkpoints/best_model.pth')
    else:
        no_improve += 1
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | loss:{rloss/tot:.4f} acc:{correct/tot:.4f} | '
          f'val_loss:{vl/vt:.4f} val_acc:{vacc:.4f} | lr:{lr_now:.6f} | {time.time()-t0:.1f}s{mark}')
    if no_improve >= patience:
        print(f'⏹ Early stopping'); break

print(f'\n🎯 Best validation accuracy: {best_acc*100:.2f}%')

In [ ]:
# Cell 6: Final Evaluation
ckpt = torch.load('checkpoints/best_model.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_p, all_l = [], []
with torch.no_grad():
    for imgs, labs in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_p.extend(preds); all_l.extend(labs.numpy())

print(f'Test Accuracy: {np.mean(np.array(all_p)==np.array(all_l))*100:.2f}%\n')
print(classification_report(all_l, all_p, target_names=CLASS_NAMES, zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(all_l, all_p))

In [ ]:
# Cell 7: Download
from google.colab import files
files.download('checkpoints/best_model.pth')
print('✅ Place in AlzheimerDetection/checkpoints/')